# Introduction
In this notebook I compare first-order time-stepping MPOs generated via two methods: one that is created by converting a dense Laplacian matrix, and the other which is created analytically according to Kazeev and Khoromskij (2012).

In [1]:
import importlib
import utils
importlib.reload(utils)

import numpy as np
import quimb.tensor as qtn

In the finite second difference scheme, the 1D Laplacian (in other words, the second derivative at a point) is approximated as
$$(\Delta u)_i=2u_i-u_{i+1}-u_{i-1}$$
where we have discretized our function space into pixels indexed by $i$, $0\leq i<2^n$.

We can hence, for the example of $n=2$, express a Laplacian matrix 

$$
\begin{aligned}
\Delta &= 2\mathbb I -S_+ - S_- \\
&= 2\begin{pmatrix}
    1 & 0 & 0 & 0 \\
    0 & 1 & 0 & 0  \\
    0 & 0 & 1 & 0  \\
    0 & 0 & 0 & 1
\end{pmatrix}
-
\begin{pmatrix}
    0 & 1 & 0 & 0 \\
    0 & 0 & 1 & 0 \\
    0 & 0 & 0 & 1  \\
    0 & 0 & 0 & 0
\end{pmatrix}
-
\begin{pmatrix}
    0 & 0 & 0 & 0 \\
    1 & 0 & 0 & 0  \\
    0 & 1 & 0 & 0  \\
    0 & 0 & 1 & 0
\end{pmatrix}
\end{aligned}
$$

The Laplacian MPO can be formed in exactly the same way, adding MPO representations of $\mathbb I, S_+, S_-$.

The time-stepping operator is given by
$$
A = \mathbb I - \text{cfl} \cdot \Delta = (1-2\ \text{cfl})\mathbb I +\text{cfl} \ S_+ + \text{cfl} \ S_-
$$

The functions below construct the MPO representation of the time-stepping operator using the above expression, building the $S_+$ and $S_-$ MPOs analytically using the paper by Kazeev and Khoromskij.
# Generation of MPO Analytically



The paper defines the following two matrices

<img src="images/identity-and-sp-matrices.png" width="30%">

and defines the $\bowtie$ operation as

<img src="images/inner-core-product.png" width="50%">

This above operation is the exact same as matrix multiplication, just written for the context of the paper's notation. This is used to connect cores in an MPO chain.

The paper then lists the MPO representation of $S_+$ as

<img src="images/shift-plus-operator.png" width="50%">

Note that each core is really a rank-4 tensor $\text{(left bond index, output bit, input bit, right bond index)}$, but since the output and input bits are guaranteed to have dimension 2, they are left implicit in the above expressions. So for example, the left core of the shift plus operator has left bond index of 1 (the number of rows) and right bond index of 2 (the number of columns). The 2x2 $I$ and $J$ (or the transpose of $J$, $J'$) matrices populate each entry as they fill the space of the input and output bits.



In [ ]:
# the paper defines
# J = [0, 1] and J' = [0, 0]
#     [0, 0]          [1, 0]

def shift_plus_cores(n: int):
    # this function generates the cores for the shift_plus MPO but returns them as a numpy array, not as an MPO
    if n < 2:
        raise ValueError("n must be > 1")
    
    I = np.array([[1.0, 0.0],
                  [0.0, 1.0]])
    J = np.array([[0.0, 1.0],
                  [0.0, 0.0]])
    JT = J.T

    # first core is [I, J]
    first_core = np.zeros((1,2,2,2))
    first_core[0,:,:,0] = I
    first_core[0,:,:,1] = J
    
    # middle cores are [I  J ]
    #                  [0  JT]
    middle_core = np.zeros((2,2,2,2))
    middle_core[0,:,:,0] = I
    middle_core[0,:,:,1] = J
    middle_core[1,:,:,1] = JT

    # last core is [J ]
    #              [JT]
    last_core = np.zeros((2,2,2,1))
    last_core[0,:,:,0] = J
    last_core[1,:,:,0] = JT

    cores = [first_core] + [middle_core.copy() for _ in range(n-2)] + [last_core]

    cores_quimb = [np.transpose(core, (0,3,1,2)) for core in cores]
    # the paper uses (left bond index, output, input, right bond index)
    # but quimb uses (left bond index, right bond index, output, input)

    return cores_quimb


def shift_plus_mpo(n: int):
    # converts the cores returned by shift_plus_cores into an MPO
    cores_quimb = shift_plus_cores(n)
    return qtn.MatrixProductOperator(cores_quimb, shape='lrud')

def shift_minus_mpo(n: int):
    # swaps the input and output indices of each core from shift_plus_cores to make it do shift minus instead
    cores_quimb = shift_plus_cores(n)
    cores_quimb_transposed = [np.transpose(core, (0,1,3,2)).copy() for core in cores_quimb]
    return qtn.MatrixProductOperator(cores_quimb_transposed, shape='lrud')

def identity_mpo(n):
    W = np.zeros((1, 1, 2, 2))
    W[0, 0] = np.eye(2)
    arrays = [W.copy() for _ in range(n)] 
    return qtn.MatrixProductOperator(arrays, shape='lrud') # l=left, r=right, u=upper/output, d=lower/input

    

def time_step_mpo(n: int, cfl: float, cutoff=1e-12, max_bond=64):
    I  = identity_mpo(n)
    Sp = shift_plus_mpo(n)
    Sm = shift_minus_mpo(n)

    A = (1 - 2 * cfl) * I + cfl * Sp + cfl * Sm
    A.compress(cutoff=cutoff, max_bond=max_bond)
    return A

# Parameters

In [3]:
# ================
# INPUT PARAMETERS
# ================

n = 5
steps = 10               # number of steps required for time evolution

def u(x):
    return np.sin(2*np.pi*2*x) + 0.5*np.sin(2*np.pi*7*x)

nu = 1e-3                 # diffusion coefficient 
save_every = 200          # after this many steps, take a snapshot of the function to plot on the graph
cfl = 0.1                 # controls time step relative to grid spacing. affects stability of time0-step scheme



# ================
# FIXED PARAMETERS
# ================

N     = 2**n
x     = np.linspace(0, 1, N, endpoint=False)

dx    = x[1] - x[0] 
dt    = cfl * dx*dx / nu 
u0    = u(x)

# Comparing Norms
Here, I generate a dense Laplacian matrix and use it to generate a dense time-step matrix `A_dense`.

I convert it to an MPO, `A_svd`, using Quimb's built-in `qtn.MatrixProductOperator.from_dense()` function.

Separately, I create an MPO analytically, `A_manual`, using the functions above in this notebook.

Then, I convert `A_svd` and `A_manual` back to dense matrices as `A_svd_densified` and `A_manual_densified` and observe that their norms are nearly identical.

**Printing `A_svd_densified` and `A_manual_densified`, we see that even up to 9dp, the matrices are identical.**

In [4]:
L = utils.laplacian(N, dx, "dirichlet", "dense")
A_dense = utils.time_step(L, 1, dt, nu)[0]
A_svd = utils.mat_to_qtt_mpo(A_dense, n)

A_manual = time_step_mpo(n, cfl)

A_svd_densified = np.asarray(A_svd.to_dense()).reshape(N,N)
A_manual_densified = np.asarray(A_manual.to_dense()).reshape(N, N)

print()
print("||A_svd - A_dense||    =", np.linalg.norm(A_svd_densified - A_dense))
print("||A_manual - A_dense|| =", np.linalg.norm(A_manual_densified - A_dense))
print("||A_manual - A_svd||   =", np.linalg.norm(A_manual_densified - A_svd_densified))

print()
print(np.round(A_svd_densified, 9))
print()
print(A_manual_densified)


||A_svd - A_dense||    = 4.27863164248793e-15
||A_manual - A_dense|| = 1.6014269617894046e-15
||A_manual - A_svd||   = 4.4258996490073184e-15

[[ 0.8  0.1  0.  ...  0.   0.   0. ]
 [ 0.1  0.8  0.1 ...  0.   0.   0. ]
 [ 0.   0.1  0.8 ...  0.   0.   0. ]
 ...
 [ 0.   0.   0.  ...  0.8  0.1 -0. ]
 [ 0.   0.   0.  ...  0.1  0.8  0.1]
 [ 0.   0.   0.  ...  0.   0.1  0.8]]

[[0.8 0.1 0.  ... 0.  0.  0. ]
 [0.1 0.8 0.1 ... 0.  0.  0. ]
 [0.  0.1 0.8 ... 0.  0.  0. ]
 ...
 [0.  0.  0.  ... 0.8 0.1 0. ]
 [0.  0.  0.  ... 0.1 0.8 0.1]
 [0.  0.  0.  ... 0.  0.1 0.8]]


# Time Evolution by MPO Contraction

Here, I run time evolution of the input MPS `mps0` using both `A_svd` and `A_manual`.

**Time taken for time evolution using `A_manual` is significantly larger and the final states of each MPS completely disagree.**

Time evolution is done by repeatedly performing MPS-MPO contraction between the current state MPS and the time-step MPO and compressing via SVD truncation and limiting bond dimension. From the table, it can be seen that the majority of the time taken in evolving `A_manual` comes from compression.

In [5]:
mps0 = utils.vec_to_qtt_mps(u0, n)

print("SVD time evolution:")
saved_svd, bonds_svd = utils.evolve_mps_timed(mps0, [A_svd,], steps)

print()
print("Manual time evolution:")
saved_manual, bonds_manual = utils.evolve_mps_timed(mps0, [A_manual,], steps)

final_state_svd = np.asarray(saved_svd[-1].to_dense()).reshape(-1)
final_state_manual = np.asarray(saved_manual[-1].to_dense()).reshape(-1)

print()
print("||mps_svd_last - mps_manual_last|| =", np.linalg.norm(final_state_svd - final_state_manual))

SVD time evolution:
st | apply  |compress| total  | bond
----------------------------------------
 0 | 0.0010 | 0.0064 | 0.0073 |   4
 1 | 0.0010 | 0.0004 | 0.0014 |   4
 2 | 0.0005 | 0.0004 | 0.0009 |   4
 3 | 0.0004 | 0.0003 | 0.0007 |   4
 4 | 0.0003 | 0.0003 | 0.0007 |   4
 5 | 0.0003 | 0.0003 | 0.0007 |   4
 6 | 0.0003 | 0.0003 | 0.0007 |   4
 7 | 0.0003 | 0.0003 | 0.0007 |   4
 8 | 0.0003 | 0.0005 | 0.0008 |   4
 9 | 0.0005 | 0.0004 | 0.0008 |   4

Manual time evolution:
st | apply  |compress| total  | bond
----------------------------------------
 0 | 0.0008 | 0.0005 | 0.0014 |  12
 1 | 0.0007 | 0.0012 | 0.0019 |  36
 2 | 0.0011 | 0.0543 | 0.0554 |  64
 3 | 0.0025 | 0.0817 | 0.0841 |  64
 4 | 0.0051 | 0.1741 | 0.1792 |  64
 5 | 0.0127 | 0.3433 | 0.3561 |  64
 6 | 0.0370 | 0.8710 | 0.9080 |  64
 7 | 0.0779 | 1.0838 | 1.1617 |  64
 8 | 0.2500 | 3.1177 | 3.3676 |  64
 9 | 0.5075 | 10.2855 | 10.7930 |  64

||mps_svd_last - mps_manual_last|| = 2.0431750672137055


# Time Evolution using Dense Matrices
Here, I created three copies of the initial discretised function `u0` and then do time evolution by the 'normal' way.

I do three versions: `u_orig` evolves using the very first time-step operator generated using the dense Laplacian matrix, `u_svd` evolves using the SVD MPO that was converted to a matrix, and `u_manual` evolves using the analytical MPO that was converted to a matrix.

Although the final states completely disagreed when evolved via the TN method, **they are now almost equal when the MPOs were converted to matrices.**

In [6]:
u_orig   = u0.copy()
u_svd    = u0.copy()
u_manual = u0.copy()

for _ in range(steps):
    u_orig   = A_dense @ u_orig
    u_svd    = A_svd_densified @ u_svd
    u_manual = A_manual_densified @ u_manual

final_state_orig         = np.asarray(u_orig)
final_state_svd_dense    = np.asarray(u_svd)
final_state_manual_dense = np.asarray(u_manual)

print("||orig - svd||    =", np.linalg.norm(final_state_orig - final_state_svd_dense))
print("||orig - manual|| =", np.linalg.norm(final_state_orig - final_state_manual_dense))
print("||svd - manual||  =", np.linalg.norm(final_state_svd_dense - final_state_manual_dense))

||orig - svd||    = 3.0899389294987514e-14
||orig - manual|| = 1.0765006800971721e-14
||svd - manual||  = 3.142778255632212e-14


# Conclusion
The fact that the Laplacian matrix is displayed correctly and that the final states when evolved via matrix form produce the same states tells me that my analytical implementation of the time-step MPO is correct.

The fact that the state diverges greatly when evolved via TN method tells me that the MPO may not be stored in as optimal a format as when Quimb converts a dense matrix to an MPO.

# Reference

V. A. Kazeev and B. N. Khoromskij, SIAM Journal on Matrix Analysis and Applications 33, 742 (2012).